In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
df = pd.read_csv('/content/drive/MyDrive/monkeytype/monkeytyping.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6159 entries, 0 to 6158
Data columns (total 31 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   participant_id             6159 non-null   object 
 1   date                       6159 non-null   object 
 2   original_filename          6159 non-null   object 
 3   trial_number               6159 non-null   int64  
 4   reaction_time              6159 non-null   object 
 5   correct                    6159 non-null   int64  
 6   correct_position           6159 non-null   object 
 7   chosen_position            6159 non-null   object 
 8   prompt                     0 non-null      float64
 9   sample_name                0 non-null      float64
 10  file_1_name                6159 non-null   object 
 11  file_2_name                6159 non-null   object 
 12  file_3_name                6139 non-null   object 
 13  file_4_name                6139 non-null   objec

In [3]:
df_symbols = df[df['colored'].isna()].copy(deep=True)
df_symbols = df_symbols[df_symbols['file_3_name'].notna()].copy(deep=True)
df_symbols = df_symbols[df_symbols['chosen_position']!='No decision'].copy(deep=True)

In [4]:
df_symbols['session_id'] = (df_symbols['trial_number'] == 1).cumsum()

In [5]:
result = (
    df_symbols
    .groupby('session_id')['chosen_position']
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index(name='most_popular_position')
)

In [6]:
df_symbols = df_symbols.merge(result, on='session_id', how='left')

In [7]:
for index, row in df_symbols.iterrows():
  df_symbols.loc[index,'chosen_position']=df_symbols.loc[index,'chosen_position'][0]
  df_symbols.loc[index,'most_popular_position']=df_symbols.loc[index,'most_popular_position'][0]

In [8]:
#df_symbols = df_symbols[df_symbols['participant_id']=='Jupiter']

In [9]:
X = df_symbols#.drop(columns=['chosen_position'])
y = df_symbols['chosen_position']
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4675 entries, 0 to 4674
Data columns (total 33 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   participant_id             4675 non-null   object 
 1   date                       4675 non-null   object 
 2   original_filename          4675 non-null   object 
 3   trial_number               4675 non-null   int64  
 4   reaction_time              4675 non-null   object 
 5   correct                    4675 non-null   int64  
 6   correct_position           4675 non-null   object 
 7   chosen_position            4675 non-null   object 
 8   prompt                     0 non-null      float64
 9   sample_name                0 non-null      float64
 10  file_1_name                4675 non-null   object 
 11  file_2_name                4675 non-null   object 
 12  file_3_name                4675 non-null   object 
 13  file_4_name                4675 non-null   objec

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,        # 20% — тест
    random_state=0,      # фиксируем для воспроизводимости
    stratify=y            # важно для логистической регрессии!
)

In [11]:
result = (
    X_train
    .groupby('session_id')['chosen_position']
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index(name='most_popular_position2')
)

In [12]:
X_train = X_train.merge(result, on='session_id', how='left')

In [13]:
X_test = X_test.merge(
    X_train[['session_id', 'most_popular_position2']].drop_duplicates(),
    on='session_id',
    how='left'
)

In [14]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = X_test['most_popular_position2']
print("Accuracy of baseline classifier:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy of baseline classifier: 0.4909090909090909
              precision    recall  f1-score   support

           1       0.40      0.04      0.07        98
           2       0.49      0.80      0.61       421
           3       0.52      0.34      0.41       331
           4       0.25      0.07      0.11        85

    accuracy                           0.49       935
   macro avg       0.42      0.31      0.30       935
weighted avg       0.47      0.49      0.44       935

